# ResNet18

In [1]:
import torch
import torch.nn as nn

In [9]:
class BasicBlock(nn.Module):
  '''
  input: (N, C, H, W)
  output:
   first_layer: (N, C, H, W)
   other layers: (N 2*C, H/2, W/2)
  '''
  def __init__(self, in_channels, out_channels, first_stride=2):
    super(BasicBlock, self).__init__()
    self.first_stride=2
    self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=first_stride, padding=1)
    self.bn1 = nn.BatchNorm2d(out_channels)
    self.conv2 = nn.Conv2d(out_channels, out_channels, 3, stride=1, padding=1)
    self.bn2 = nn.BatchNorm2d(out_channels)

    self.relu = nn.ReLU(inplace=True)

    self.downsample = None
    # downsample is not needed for the first layer
    if first_stride != 1:
      self.downsample = nn.Sequential(
          nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=first_stride, bias=False),
          nn.BatchNorm2d(out_channels),
      )

  def forward(self, x):
    identity = x
    x = self.conv1(x)
    x = self.bn1(x)
    x = self.relu(x)
    x = self.conv2(x)
    x = self.bn2(x)

    if self.downsample:
      identity = self.downsample(identity)

    x = x + identity
    x = self.relu(x)

    return x


In [16]:
class ResNet18(nn.Module):
  def __init__(self):
    super(ResNet18, self).__init__()
    self.conv = nn.Conv2d(in_channels=3, out_channels=64, kernel_size=7, stride=2, padding=3, bias=False)
    self.bn = nn.BatchNorm2d(64)
    self.relu = nn.ReLU(inplace=True)
    self.max_pool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

    self.layer_1 = BasicBlock(64, 64, first_stride=1)
    self.layer_2 = BasicBlock(64, 128)
    self.layer_3 = BasicBlock(128, 256)
    self.layer_4 = BasicBlock(256, 512)

    self.avgpool = nn.AdaptiveAvgPool2d((1, 1))


  def forward(self, x):
    # (N, 3, 224, 224)
    x = self.conv(x)
    # (N, 64, 112, 112)
    x = self.relu(self.bn(x))
    # (N, 64, 112, 112)
    x = self.max_pool(x)
    # (N, 64, 56, 56)

    x = self.layer_1(x)
    # (N, 64, 56, 56)
    x = self.layer_2(x)
    # (N, 128, 28, 28)
    x = self.layer_3(x)
    # (N, 256, 14, 14)
    x = self.layer_4(x)
    # (N, 512, 7, 7)

    x = self.avgpool(x)
    # (N, 512, 1, 1)

    return x

In [14]:
model = ResNet18()

In [15]:
sample_input = torch.randn(1, 3, 224, 224)
sample_output = model(sample_input)
print(sample_output.shape)

torch.Size([1, 512, 1, 1])
